In [ ]:
import pandas as pd
from pathlib import Path


# List of CSV files with their corresponding columns to extract
baby_health_files = [
    {'file': 'CBA.csv', 'columns': ['CBAA04a', 'CBAA04b', 'CBAA04c', 'CBAA05a_Gm']},
    {'file': 'pregnancy_outcomes.csv', 'columns': ['TYPE_CA_A09', 'GAwksCA']},
]

# Initialize an empty DataFrame to store the extracted data
merged_data = pd.DataFrame()
data_dir = Path('../Data')

# Iterate over each CSV file
for csv_data in baby_health_files:
    path = data_dir / csv_data['file']
    df = pd.read_csv(path, encoding='latin-1', low_memory=False)
    
    missing = [col for col in csv_data['columns'] if col not in df.columns]
    if missing:
        raise KeyError(f"{path.name} is missing columns: {missing}")
    
    # Concatenate the extracted data with the existing data
    df = df[csv_data['columns']]
    merged_data = pd.concat([merged_data, df], axis=1)

merged_data['CBAA05a_Gm'] = pd.to_numeric(merged_data['CBAA05a_Gm'], errors='coerce')

merged_data['Birth_Weight_encoded'] = pd.cut(merged_data['CBAA05a_Gm'],
                                    bins=[0, 1000, 1500, 2500, float('inf')],
                                    labels=[0, 1, 2, 3])
merged_data['Preterm_Birth_encoded'] = pd.cut(merged_data['GAwksCA'],
                                        bins=[0, 28, 32, 37],
                                        labels=[0, 1, 2])

# Save the merged data to a new CSV file
merged_data.to_csv('../Data/merged_baby_health_data.csv', index=False)


In [ ]:
from sklearn.linear_model import LogisticRegression

# Load the data for each group of variables
df_baby_health = pd.read_csv('../Data/merged_baby_health_data.csv')
# df_health = pd.read_csv('health.csv')
# df_lifestyle = pd.read_csv('lifestyle.csv')

# Define the outcome variable
outcome = 'mental_health_outcome'

# Define the predictor variable groups
demographic_vars = ['age', 'gender', 'education', 'income']
health_vars = ['blood_pressure', 'cholesterol', 'heart_rate']
lifestyle_vars = ['exercise_frequency', 'diet_quality', 'smoking_status']

# Create separate subsets of data for each group
df_demographic_subset = df_demographic[[outcome] + demographic_vars]
df_health_subset = df_health[[outcome] + health_vars]
df_lifestyle_subset = df_lifestyle[[outcome] + lifestyle_vars]

# Function to train the logistic regression model and print the results
def train_logistic_regression(df_subset):
    X = df_subset.drop(outcome, axis=1)
    y = df_subset[outcome]

    model = LogisticRegression()
    model.fit(X, y)

    print(f"Model coefficients: {model.coef_}")
    print(f"Model intercept: {model.intercept_}")
    print(f"Model accuracy: {model.score(X, y)}")

# Train and evaluate the logistic regression model for each group
print("Demographic Variables:")
train_logistic_regression(df_demographic_subset)

print("Health Variables:")
train_logistic_regression(df_health_subset)

print("Lifestyle Variables:")
train_logistic_regression(df_lifestyle_subset)


NameError: name 'df_demographic' is not defined